# 05 · CASE WHEN and denial rates

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

| Table | Columns |
|---|---|
| `claims` (1,000) | `claim_id`, `patient_id`, `provider_id`, `date_of_service`, `cpt_code`, `billed_amount`, `paid_amount`, `claim_status`, `denial_reason` |
| `patients` (200) | `patient_id`, `patient_name`, `date_of_birth`, `insurance_plan`, `state` |
| `providers` (13) | `provider_id`, `provider_name`, `specialty`, `state` |
| `procedures` (13) | `cpt_code`, `description`, `service_type`, `standard_charge` |

> 🇪🇸 **Para Josué:** este es el tema **más importante del proyecto**. La **tasa de rechazo** (*denial rate*) es el número que más mira un equipo de billing, y es el corazón de tu análisis final.
> La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


---
## Example · CASE WHEN to label rows

Put each claim in a **work queue** depending on its status.

> 🇪🇸 **Te recuerdo:** `CASE WHEN` es el `SI()` de Excel. Revisa las condiciones **en orden** y se queda con la primera que se cumple; `ELSE` es "si ninguna se cumplió". Siempre termina con **`END`**.

In [2]:
duckdb.sql("""
    SELECT claim_id, claim_status,
           CASE WHEN claim_status = 'Paid' THEN 'Closed'
                WHEN claim_status = 'Denied' THEN 'Needs review'
                ELSE 'Waiting'
           END AS work_queue
    FROM claims
    ORDER BY claim_id
""").df()

,claim_id,claim_status,work_queue
0,CL00001,Paid,Closed
1,CL00002,Paid,Closed
2,CL00003,Denied,Needs review
3,CL00004,Paid,Closed
4,CL00005,Paid,Closed
...,...,...,...
995,CL00996,Paid,Closed
996,CL00997,Paid,Closed
997,CL00998,Paid,Closed
998,CL00999,Paid,Closed


## Exercise 1 · Claims by amount tier

Classify each claim by its `billed_amount`:
- **`High`**: 1,000 or more
- **`Medium`**: 300 or more (and less than 1,000)
- **`Low`**: everything else

Then show, for each tier, the number of claims and the total billed, from the most to the fewest claims.

Columns: `amount_tier`, `total_claims`, `total_billed`.

✔ **3 rows:** Low · 672 · $106,411 / High · 189 · $258,231 / Medium · 139 · $60,443.

> 🇪🇸 Primero el `CASE WHEN` con 3 resultados (como el ejemplo), con `AS amount_tier`. Después `GROUP BY amount_tier`, como en el tema 2.
> **Ojo con el orden de las condiciones:** pon primero `>= 1000` y después `>= 300`. Si pones `>= 300` primero, un claim de 1,500 se quedaría en "Medium", porque `CASE` se queda con la primera que se cumple.
> Fíjate en el resultado: solo 189 claims son *High*, pero suman más dinero que los 672 *Low*.

In [18]:
duckdb.sql("""
    SELECT 
    COUNT(*) AS total_claims,
    SUM(billed_amount) AS total_billed,
    CASE WHEN billed_amount >= 1000 THEN 'High'
    WHEN billed_amount >= 300 THEN 'Medium'
    ELSE 'Low'
    END AS amount_tier 
    FROM claims
    GROUP BY amount_tier
    ORDER BY total_claims DESC
""").df()

,total_claims,total_billed,amount_tier
0,672,106411.0,Low
1,189,258231.0,High
2,139,60443.0,Medium


---
## Example · The denial rate

Count the denied claims with `SUM(CASE WHEN ...)`, and divide by all the claims.

> 🇪🇸 **Te recuerdo:**
> - `CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END` pone **1** si el claim es Denied y **0** si no. `SUM` suma esos unos = cuántos Denied hay.
> - **Tasa** = rechazados ÷ total × 100. Se escribe `100.0` (con punto) para que SQL saque decimales.
> - `ROUND(..., 1)` = un decimal.

**Example:** the denial rate of provider PR002.

In [4]:
duckdb.sql("""
    SELECT COUNT(*) AS total_claims,
           SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) AS denied_claims,
           ROUND(100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS denial_rate
    FROM claims
    WHERE provider_id = 'PR002'
""").df()

,total_claims,denied_claims,denial_rate
0,85,20.0,23.5


> 🇪🇸 Léelo así: PR002 tiene 85 claims, 20 rechazados → 20 ÷ 85 × 100 = **23.5%**.

## Exercise 2 · Overall denial rate

Show **one row** with the denial rate of **all** the claims.

Columns: `total_claims`, `denied_claims`, `denial_rate`.

✔ **1 row:** 1,000 · 174 · **17.4**.

> 🇪🇸 Como el ejemplo, pero sin el `WHERE`: quieres todos los claims. Este 17.4% es tu **línea base**: con este número vas a comparar todo lo demás.

In [20]:
duckdb.sql("""
    SELECT COUNT(*) AS total_claims,
               SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) AS denied_claims,
               ROUND(100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS denial_rate
        FROM claims
""").df()

,total_claims,denied_claims,denial_rate
0,1000,174.0,17.4


## Exercise 3 · Denial rate by service type ⭐

For each `service_type`, show the number of claims, the denied claims and the denial rate, from the highest to the lowest rate.

Columns: `service_type`, `total_claims`, `denied_claims`, `denial_rate`.

✔ **5 rows.** The first one is **Imaging** · 247 · 66 · **26.7**. The last one is **Office Visit** · 268 · 26 · **9.7**.

> 🇪🇸 Tres cosas que ya sabes, juntas: `JOIN` con `procedures` (código en común: `cpt_code`), la fórmula de la tasa del ejemplo y `GROUP BY service_type`.
> **Este es el hallazgo principal de tu proyecto:** Imaging se rechaza casi 3 veces más que una consulta normal.

In [32]:
duckdb.sql("""
    SELECT pr.service_type,
    COUNT(*) AS total_claims,
    SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) AS denied_claims,
    ROUND(100.0*SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END)/ COUNT(*), 1) AS denial_rate 
    FROM claims AS c
    LEFT JOIN procedures as pr
    ON c.cpt_code = pr.cpt_code
    GROUP BY pr.service_type
    ORDER BY denial_rate DESC
    
""").df()

,service_type,total_claims,denied_claims,denial_rate
0,Imaging,247,66.0,26.7
1,Physical Therapy,198,48.0,24.2
2,Cardiology,80,11.0,13.8
3,Lab,207,23.0,11.1
4,Office Visit,268,26.0,9.7


## Exercise 4 · Denial rate by insurance plan

For each `insurance_plan`, show the number of claims, the denied claims and the denial rate, from the highest to the lowest rate.

Columns: `insurance_plan`, `total_claims`, `denied_claims`, `denial_rate`.

✔ **3 rows.** The first one is **Medicaid** · 239 · 56 · **23.4**. The last one is **Medicare** · 367 · 41 · **11.2**.

> 🇪🇸 Igual que el ejercicio 3, pero el `JOIN` es con `patients` (código en común: `patient_id`), porque ahí está `insurance_plan`.

In [33]:
duckdb.sql("""
    SELECT pa.insurance_plan,
    COUNT(*) AS total_claims,
    SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) AS denied_claims,
    ROUND(100.0*SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END)/COUNT(*),1) AS denial_rate
    FROM claims as c
    LEFT JOIN patients as pa
    ON c.patient_id = pa.patient_id
    GROUP BY pa.insurance_plan
    ORDER BY denial_rate DESC

""").df()

,insurance_plan,total_claims,denied_claims,denial_rate
0,Medicaid,239,56.0,23.4
1,Commercial,394,77.0,19.5
2,Medicare,367,41.0,11.2


---
## Exercise 5 · Status breakdown by service type (pivot)

For each `service_type`, show **in separate columns** how many claims are Paid, Denied and Pending. Sort by `service_type` (A to Z).

Columns: `service_type`, `paid_claims`, `denied_claims`, `pending_claims`.

✔ **5 rows.** The first one is **Cardiology** · 64 · 11 · 5.

> 🇪🇸 Esto es una **tabla dinámica** de Excel: los estados pasan de filas a columnas. En inglés se llama *pivot*.
> El truco: tres `SUM(CASE WHEN ...)`, uno por cada estado, cada uno con su `AS`:
> ```sql
> SUM(CASE WHEN claims.claim_status = 'Paid' THEN 1 ELSE 0 END) AS paid_claims,
> ```
> Haz lo mismo para `'Denied'` y `'Pending'`. Ojo con las mayúsculas: `'Paid'`, `'Denied'`, `'Pending'`.

In [ ]:
duckdb.sql("""
    SELECT pr.service_type,
        SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) AS denied_claims,
        SUM(CASE WHEN claim_status = 'Paid' THEN 1 ELSE 0 END) AS paid_claims,
        SUM(CASE WHEN claim_status = 'Pending' THEN 1 ELSE 0 END) AS pending_claims
        FROM claims as c
        LEFT JOIN procedures as pr
        ON c.cpt_code = pr.cpt_code
        GROUP BY service_type
        ORDER BY service_type
""").df()

,service_type,denied_claims,paid_claims,pending_claims
0,Cardiology,11.0,64.0,5.0
1,Imaging,66.0,169.0,12.0
2,Lab,23.0,175.0,9.0
3,Office Visit,26.0,219.0,23.0
4,Physical Therapy,48.0,136.0,14.0


---
## Challenge ⭐⭐⭐ · Providers above 20% denial rate

Show the providers with a denial rate **greater than 20%**, from the highest to the lowest rate. Use the **provider name**.

Columns: `provider_name`, `total_claims`, `denial_rate`.

✔ **5 rows.** The first one is **Desert Sun Radiology** · 72 · **33.3**. The last one is **Bayview Physical Therapy** · 89 · **20.2**.

> 🇪🇸 `JOIN` con `providers`, `GROUP BY provider_name` y la fórmula de la tasa. Para quedarte solo con los de más de 20%, usa **`HAVING`** (filtra grupos, porque la tasa es un dato calculado).
> **Te recuerdo:** en el `HAVING` puedes escribir el nombre de la columna (`HAVING denial_rate > 20`) o repetir la fórmula completa. En SQL Server o PostgreSQL hay que repetir la fórmula.
> Estos son los 5 proveedores que necesitan ayuda primero. ¿Qué tipo de servicio hacen casi todos?

In [40]:
duckdb.sql("""
    SELECT p.provider_name,
    COUNT(*) AS total_claims,
    ROUND(100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*),1) AS denial_rate
    FROM claims AS c
    LEFT JOIN providers as p
    ON c.provider_id = p.provider_id
    GROUP BY p.provider_name
    HAVING denial_rate > 20
    ORDER BY denial_rate DESC
""").df()

,provider_name,total_claims,denial_rate
0,Desert Sun Radiology,72,33.3
1,Harbor Rehab & Therapy,108,27.8
2,Empire Diagnostic Imaging,87,25.3
3,Lakeside Imaging,85,23.5
4,Bayview Physical Therapy,89,20.2


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. ¿Por qué en la fórmula se escribe `100.0` y no `100`? (Pista: DuckDB divide bien de todas formas, pero en **SQL Server y PostgreSQL**, que son los que vas a ver en el trabajo, `174 / 1000` da **0**, porque dividen enteros sin decimales. ¿Qué cambia al poner `100.0` adelante?)
> 2. Con los ejercicios 3 y 4, si fueras supervisor de billing y solo pudieras revisar **un** tipo de claim antes de enviarlo, ¿cuál revisarías y qué le revisarías? (Pista: recuerda el motivo de rechazo #1 del tema 2.)